# Week 2 — Fashion-MNIST Binary, CLIP image encoder (N=1000)
**Project**: Quantum Kernels — second pure-vision dataset (balances the table vs CIFAR-10)

**Dataset**: Fashion-MNIST binary — T-shirt (class 0) vs Sneaker (class 7)
A different visual domain from CIFAR-10 (clothing product photos vs natural objects),
so the vision leg shows breadth, not repetition.

**Encoder**: CLIP image encoder (512-dim)
**Note:** Fashion-MNIST is 28x28 grayscale -> converted to RGB for the encoder.
**Metric:** binary F1 | **Runtime:** ~6 min

In [ ]:
!pip install datasets transformers torch Pillow scikit-learn numpy --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage

In [ ]:
TAG = 'fashion_clip'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

---
## Step 1: Load Fashion-MNIST binary (T-shirt vs Sneaker)

In [ ]:
FASHION_CLASSES = {0: 'T-shirt', 7: 'Sneaker'}
TARGET = [0, 7]   # the two classes
import random; random.seed(42)

from datasets import load_dataset
print("Loading zalando-datasets/fashion_mnist...")
ds = load_dataset("zalando-datasets/fashion_mnist")

def collect(split, n_per):
    imgs, lbls = [], []
    c = {0:0, 7:0}
    for s in ds[split]:
        l = s['label']
        if l in TARGET and c[l] < n_per:
            imgs.append(s['image']); lbls.append(l); c[l]+=1
        if all(c[k]>=n_per for k in TARGET): break
    return imgs, lbls

train_images, train_lbl_raw = collect('train', 600)
test_images,  test_lbl_raw  = collect('test', 130)

# remap labels to 0/1 (0=T-shirt, 1=Sneaker) and balance/shuffle
def remap_balance(imgs, lbls, n_per):
    pairs = [(im, 0 if l==0 else 1) for im,l in zip(imgs,lbls)]
    c0 = [p for p in pairs if p[1]==0][:n_per]
    c1 = [p for p in pairs if p[1]==1][:n_per]
    comb = c0+c1; random.shuffle(comb)
    return [p[0] for p in comb], np.array([p[1] for p in comb])

train_images, train_labels = remap_balance(train_images, train_lbl_raw, 550)
test_images,  test_labels  = remap_balance(test_images,  test_lbl_raw, 110)
print(f"Train {len(train_images)} | Test {len(test_images)}")
print(f"Balance: train {np.mean(train_labels):.1%} sneaker")

# preview
fig, axes = plt.subplots(1, 6, figsize=(12,2.5))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i], cmap='gray'); ax.set_title({0:'T-shirt',1:'Sneaker'}[train_labels[i]], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

---
## Step 2: Extract CLIP image encoder embeddings — checkpoint after

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
from transformers import CLIPModel, CLIPProcessor
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE); model.eval()
print("CLIP loaded (512-dim).")

def extract(images, batch_size=32):
    embs=[]
    for i in range(0,len(images),batch_size):
        # grayscale -> RGB
        batch=[im.convert('RGB') for im in images[i:i+batch_size]]
        pv = processor(images=batch, return_tensors='pt')['pixel_values'].to(DEVICE)
        with torch.no_grad():
            vis = model.vision_model(pixel_values=pv).pooler_output
            e = model.visual_projection(vis)
            e = e / torch.linalg.norm(e, dim=-1, keepdim=True)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)

t0=time.time()
train_emb = extract(train_images); test_emb = extract(test_images)
print(f"Done {time.time()-t0:.0f}s | train {train_emb.shape} test {test_emb.shape}")
assert train_emb.shape[1] == 512
checkpoint('fashion_clip_raw.npz', lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                                               train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA + normalize

In [ ]:
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in [4,8,12,16]}
print("PCA variance (512-dim -> reduced):")
for n in [4,8,12,16]: print(f"  {n:>2} -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")
pca_8 = pca_objects[8]
scaler = MinMaxScaler(feature_range=(0, np.pi)); scaler.fit(pca_8.transform(train_emb))
X_train_scaled = np.clip(scaler.transform(pca_8.transform(train_emb)), 0, np.pi)
X_test_scaled  = np.clip(scaler.transform(pca_8.transform(test_emb)),  0, np.pi)
checkpoint('fashion_clip_pca_scaler.pkl', lambda p: pickle.dump(
    {'pca':pca_8,'scaler':scaler,'n_qubits':8,'embedding_type':'clip_512','n_classes':2}, open(p,'wb')))

---
## Step 4: Classical baselines + save

In [ ]:
idx_tr=np.random.RandomState(42).choice(len(X_train_scaled),1000,replace=False)
idx_te=np.random.RandomState(42).choice(len(X_test_scaled),200,replace=False)
X_tr,y_tr=X_train_scaled[idx_tr],train_labels[idx_tr]
X_te,y_te=X_test_scaled[idx_te],test_labels[idx_te]
def bf1(yt,yp): return float(f1_score(yt,yp,average='binary'))
classifiers={'Linear':SVC(kernel='linear',C=1.0,random_state=42),
             'RBF':SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42),
             'Polynomial':SVC(kernel='poly',C=1.0,degree=3,coef0=1,random_state=42)}
results={}
print(f"{'Kernel':<14}{'Acc':>10}{'F1':>10}")
for name,clf in classifiers.items():
    clf.fit(X_tr,y_tr); yp=clf.predict(X_te)
    results[name]={'accuracy':float(accuracy_score(y_te,yp)),'f1':bf1(y_te,yp)}
    print(f"{name:<14}{results[name]['accuracy']:>10.4f}{results[name]['f1']:>10.4f}")
checkpoint('fashion_clip_baseline_results.pkl', lambda p: pickle.dump(results,open(p,'wb')))
def save_final(p):
    np.savez(p, X_train=X_train_scaled,X_test=X_test_scaled,y_train=train_labels,y_test=test_labels,
             X_tr_subset=X_tr,X_te_subset=X_te,y_tr_subset=y_tr,y_te_subset=y_te)
checkpoint('fashion_clip_embeddings.npz', save_final)
loaded=np.load(_find('fashion_clip_embeddings.npz'))
assert loaded['X_tr_subset'].shape==(1000,8); assert loaded['X_te_subset'].shape==(200,8)
print("VERIFIED (1000,8)/(200,8).")
checkpoint('fashion_clip_meta.json', lambda p: json.dump(
    {'dataset':'fashion_mnist_binary','classes':'tshirt_vs_sneaker','encoder':'clip',
      'embedding_type':'clip_512','n_classes':2,'metric':'binary','baselines':results}, open(p,'w'),indent=2))
print("\nWeek 2 fashion_clip done.")

---
## Done — Fashion-MNIST CLIP image encoder

### Saved (tag = `fashion_clip`):
raw, pca_scaler, baseline_results, embeddings, meta — all in Drive.

### For Week 3/4:
Reuse `week3c_cifar10` / `week4c_cifar10`, change the loaded filename to
`fashion_clip_embeddings.npz` and the baseline file to `fashion_clip_baseline_results.pkl`.
Binary F1, same IQP kernel.

### Why this balances the table:
Second pure-vision dataset in a different domain (clothing vs natural objects),
run on the same two encoders as CIFAR-10 (CLIP + DINOv2). Vision now has two
datasets x two encoders, matching the depth of your text leg.